In [1]:
import numpy as np
import pandas as pd
from math import sqrt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import json
import xgboost as xgb
from catboost import CatBoostRegressor
from tensorflow.keras.models import load_model

In [2]:
def mape(y_true, y_pred):
    y_true = np.array(y_true).reshape(-1)
    y_pred = np.array(y_pred).reshape(-1)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def create_sequences(X, lookback=24):
    Xs = []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
    return np.array(Xs)

In [3]:
df = pd.read_csv("../data/processed/final_dataset2.csv", parse_dates=["timestamp"])

with open("../saved_models/features.json") as f:
    feature_names = json.load(f)

target = "load_actual"

In [4]:

train_size = int(len(df) * 0.8)
test_df = df.iloc[train_size:]

X_test_raw = test_df[feature_names]
y_test = test_df[target].values

In [5]:
lookback = 24

def prepare_dl_test(model_name):
    """
    Loads scalers for the given model, scales X_test_raw,
    and returns DL-ready sequences.
    """

    scaler_X_path = f"../saved_models/{model_name}_scaler_X.pkl"
    scaler_y_path = f"../saved_models/{model_name}_scaler_y.pkl"

    scaler_X = joblib.load(scaler_X_path)
    scaler_y = joblib.load(scaler_y_path)

    X_scaled = scaler_X.transform(X_test_raw)
    y_scaled = scaler_y.transform(y_test.reshape(-1,1))

    X_seq = create_sequences(X_scaled, lookback)
    y_seq = y_scaled[lookback:]  # align lengths

    return X_seq, y_seq, scaler_y


In [6]:
models = {
    "XGBoost": {
        "type": "xgb",
        "path": "../saved_models/xgb_model.json"
    },
    "CatBoost": {
        "type": "cat",
        "path": "../saved_models/catboost_model.cbm"
    },
    "RandomForest": {
        "type": "pickle",
        "path": "../saved_models/rf_model.pkl"
    },
    "SVM": {
        "type": "pickle",
        "path": "../saved_models/svm_model.pkl"
    },
    "ARIMA": {
        "type": "arima",
        "path": "../saved_models/arima_model.pkl"
    },


    # Deep learning models
    "LSTM": {
        "type": "keras_dl",
        "path": "../saved_models/lstm_model.h5"
    },
    "GRU": {
        "type": "keras_dl",
        "path": "../saved_models/gru_model.h5"
    },
    "CNN": {
        "type": "keras_dl",
        "path": "../saved_models/cnn_model.h5"
    },
    "CNN_LSTM": {
        "type": "keras_dl",
        "path": "../saved_models/cnn_lstm_model.h5"
    },
    "BPNN": {
        "type": "keras_dl",
        "path": "../saved_models/bpnn_model.h5"
    }
}

In [7]:
results = []

for name, info in models.items():
    model_type = info["type"]
    path = info["path"]

    try:
        print(f"\nEvaluating {name} ...")

        # ML MODELS ======================
        if model_type == "xgb":
            model = xgb.XGBRegressor()
            model.load_model(path)
            preds = model.predict(X_test_raw)

        elif model_type == "cat":
            model = CatBoostRegressor()
            model.load_model(path)
            preds = model.predict(X_test_raw)

        elif model_type == "pickle":
            model = joblib.load(path)
            preds = model.predict(X_test_raw)
        
        elif model_type == "arima":
            from statsmodels.tsa.arima.model import ARIMAResults
            model = ARIMAResults.load(path)
            preds = model.forecast(steps=len(X_test_raw))

        # DL MODELS ======================
        elif model_type == "keras_dl":

            dl_name = name.lower()

            if dl_name == "bpnn":
                # BPNN DOES NOT USE SEQUENCES
                scaler_X = joblib.load("../saved_models/bpnn_scaler_X.pkl")
                scaler_y = joblib.load("../saved_models/bpnn_scaler_y.pkl")

                X_scaled = scaler_X.transform(X_test_raw)
                y_scaled = scaler_y.transform(y_test.reshape(-1, 1))

                model = load_model(path, compile=False)
                pred_scaled = model.predict(X_scaled)
                preds = scaler_y.inverse_transform(pred_scaled)

                y_test_use = y_test

            else:
                # ALL OTHER DL MODELS USE SEQUENCES
                X_seq_test, y_seq_test, scaler_y = prepare_dl_test(dl_name)

                model = load_model(path, compile=False)
                pred_scaled = model.predict(X_seq_test)
                preds = scaler_y.inverse_transform(pred_scaled)

                y_test_use = y_test[lookback:]


        # For ML models, y_test is unchanged
        if model_type != "keras_dl":
            y_test_use = y_test

        # Compute metrics
        mae = mean_absolute_error(y_test_use, preds)
        rmse = sqrt(mean_squared_error(y_test_use, preds))
        mape_val = mape(y_test_use, preds)

        results.append({
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE": mape_val
        })

    except Exception as e:
        print(f"❌ Model {name} failed to evaluate: {e}")



Evaluating XGBoost ...

Evaluating CatBoost ...

Evaluating RandomForest ...

Evaluating SVM ...


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SVR was fitted without feature names
  warnings.warn(



Evaluating ARIMA ...


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(



Evaluating LSTM ...
548/548 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step

Evaluating GRU ...
548/548 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step

Evaluating CNN ...
548/548 ━━━━━━━━━━━━━━━━━━━━ 1s 998us/step

Evaluating CNN_LSTM ...
548/548 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Evaluating BPNN ...
548/548 ━━━━━━━━━━━━━━━━━━━━ 0s 630us/step


In [8]:

results_df = pd.DataFrame(results).sort_values("MAPE").reset_index(drop=True)
print("\n=== FINAL MODEL PERFORMANCE COMPARISON ===")
results_df


=== FINAL MODEL PERFORMANCE COMPARISON ===


,Model,MAE,RMSE,MAPE
0,CNN_LSTM,532.789656,693.459358,5.555090
1,RandomForest,550.102206,713.966551,5.798425
2,GRU,604.334739,760.014066,6.286093
3,LSTM,610.338968,762.401751,6.292700
4,CNN,597.287200,784.388637,6.323021
5,XGBoost,777.538782,968.765610,8.185202
6,CatBoost,837.989334,1031.833373,8.809818
7,BPNN,890.044279,1087.493293,9.455628
8,SVM,1121.918361,1341.001202,11.771882
9,ARIMA,1174.048091,1430.247139,12.855821


In [11]:
results_df.to_csv("results/model_comparison.csv", index=False)


In [9]:
for name, info in models.items():
    print("\n============")
    print(f"Evaluating {name}...")
    try:
        # replicate your evaluation logic
        print("OK")
    except Exception as e:
        print(f"FAILED: {e}")



Evaluating XGBoost...
OK

Evaluating CatBoost...
OK

Evaluating RandomForest...
OK

Evaluating SVM...
OK

Evaluating ARIMA...
OK

Evaluating LSTM...
OK

Evaluating GRU...
OK

Evaluating CNN...
OK

Evaluating CNN_LSTM...
OK

Evaluating BPNN...
OK
